In [4]:
# Table 5 reconstruction using the shared reasoning-analysis pipeline.
# This cell uses the same extractor, tokenizer, lexicons, and vocabulary
# matcher as Appendix Tables A2-A7. It writes derived outputs only when run.

from pathlib import Path
import sys
import numpy as np
import pandas as pd

def find_reasoning_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Analyses" / "Reasoning Analysis",
        Path.cwd().parent / "Reasoning Analysis",
    ]
    for candidate in candidates:
        if (candidate / "reasoning_analysis_core.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate reasoning_analysis_core.py")

REASONING_DIR = find_reasoning_dir()
if str(REASONING_DIR) not in sys.path:
    sys.path.insert(0, str(REASONING_DIR))

from reasoning_analysis_core import (
    bh_qvalues,
    bootstrap_diff_p,
    bootstrap_metric,
    load_model_rationales,
    make_doc_feature_table,
    summarize_by_model_option,
)

N_BOOT = 10000
SEED = 42

BASELINE_FILES = [
    "aya-expanse-32b_test.csv",
    "gemma3-12b_test.csv",
    "gemma3-27b_test.csv",
    "gpt4o_test.csv",
    "llama3.1_test.csv",
    "llama3.2-latest_test.csv",
    "llama3.3-latest_test.csv",
    "mistral-small3.1-latest_test.csv",
    "mistral-small-3.2-24b_test.csv",
    "qwen2.5-14b_test.csv",
    "qwen2.5-32b_test.csv",
    "qwen2.5-72b_test.csv",
]

TABLE_ROWS = [
    ("A", "Confidence / 100w", "confidence_rate_per100w"),
    ("A", "Hedging / 100w", "hedge_rate_per100w"),
    ("A", "Avg. words", "avg_words"),
    ("B", "Confidence / 100w", "confidence_rate_per100w"),
    ("B", "Epistemic / 1000w", "epistemic_terms_per1000w"),
    ("B", "Avg. words", "avg_words"),
    ("C", "Confidence / 100w", "confidence_rate_per100w"),
    ("C", "Relational / 1000w", "relational_terms_per1000w"),
    ("C", "Avg. words", "avg_words"),
]

extracted = load_model_rationales(BASELINE_FILES, data_dir=REASONING_DIR)
doc_df = make_doc_feature_table(extracted)
summary_df = summarize_by_model_option(extracted)

summary_cols = [
    "model", "option", "n_reasonings", "avg_words",
    "confidence_rate_per100w", "hedge_rate_per100w",
    "epistemic_terms_per1000w", "relational_terms_per1000w",
]
summary_metrics = summary_df[summary_cols].copy()
summary_path = REASONING_DIR / "reasoning_metrics_by_model_option_recomputed.csv"
summary_metrics.to_csv(summary_path, index=False)
print(f"[OK] Saved: {summary_path}")

table_rows = []
pvals = []

for i, (opt, metric_label, metric_key) in enumerate(TABLE_ROWS):
    sub = (
        summary_metrics[summary_metrics["option"] == opt][["model", metric_key]]
        .sort_values(metric_key, ascending=False)
        .reset_index(drop=True)
    )
    if len(sub) < 2:
        raise RuntimeError(f"Not enough models for option {opt} and metric {metric_key}")

    best_model = sub.loc[0, "model"]
    second_model = sub.loc[1, "model"]

    g1 = doc_df[(doc_df["model"] == best_model) & (doc_df["choice"] == opt)]
    g2 = doc_df[(doc_df["model"] == second_model) & (doc_df["choice"] == opt)]

    mean1, lo1, hi1 = bootstrap_metric(g1, metric_key, n_boot=N_BOOT, seed=100 + i)
    mean2, lo2, hi2 = bootstrap_metric(g2, metric_key, n_boot=N_BOOT, seed=200 + i)
    p = bootstrap_diff_p(g1, g2, metric_key, n_boot=N_BOOT, seed=300 + i)

    pvals.append(p)
    table_rows.append({
        "Option": opt,
        "Metric": metric_label,
        "Best model": best_model,
        "Best mean": mean1,
        "Best CI low": lo1,
        "Best CI high": hi1,
        "Second model": second_model,
        "Second mean": mean2,
        "Second CI low": lo2,
        "Second CI high": hi2,
        "p": p,
    })

table_df = pd.DataFrame(table_rows)
table_df["q"] = bh_qvalues(table_df["p"].to_list())
table_df["Sig. (q<.05)"] = np.where(table_df["q"] < 0.05, "Yes", "No")

for col in [
    "Best mean", "Best CI low", "Best CI high",
    "Second mean", "Second CI low", "Second CI high",
    "p", "q",
]:
    table_df[col] = table_df[col].round(3)

table_path = REASONING_DIR / "table5_bootstrap_results.csv"
table_df.to_csv(table_path, index=False)
print(f"[OK] Saved: {table_path}")
print()
print(table_df)

print("\nLaTeX rows:\n")
for _, r in table_df.iterrows():
    print(
        f"{r['Option']} & {r['Metric']} & {r['Best model']} & "
        f"\\textbf{{{r['Best mean']:.3f}}} [{r['Best CI low']:.3f}, {r['Best CI high']:.3f}] & "
        f"{r['Second model']} & {r['Second mean']:.3f} [{r['Second CI low']:.3f}, {r['Second CI high']:.3f}] & "
        f"{r['Sig. (q<.05)']} \\\\"
    )


[OK] Saved: reasoning_metrics_by_model_option_recomputed.csv
[OK] Saved: table5_bootstrap_results.csv

  Option              Metric               Best model  Best mean  Best CI low  \
0      A   Confidence / 100w          aya-expanse-32b      1.358        0.529   
1      A      Hedging / 100w  mistral-small3.1-latest      2.424        0.000   
2      A          Avg. words          aya-expanse-32b     28.048       25.905   
3      B   Confidence / 100w  mistral-small3.1-latest      1.635        0.557   
4      B   Epistemic / 1000w              qwen2.5-14b     45.802       18.382   
5      B          Avg. words          aya-expanse-32b     26.786       24.786   
6      C   Confidence / 100w          llama3.2-latest      1.521        0.840   
7      C  Relational / 1000w                    gpt4o     55.821       43.117   
8      C          Avg. words          aya-expanse-32b     26.329       25.100   

   Best CI high             Second model  Second mean  Second CI low  \
0         2.22